In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from src.db import DATA_RAW, MODELS_DIR
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from catboost import CatBoostRegressor, Pool

df = pd.read_parquet(DATA_RAW / "cars.parquet")
print("shape:", df.shape)

# ── фичи ──────────────────────────────────────────────────────────────
target = "price"

# числовые
num_features = [
    "car_age", "mileage_km", "mileage_per_year", "power_hp",
    "engine_volume_l", "owners_count", "photos_count",
    "seller_age_years", "year",
]

# булевы
bool_features = df.select_dtypes(include="bool").columns.tolist()

# категориальные
cat_features = [
    "brand", "model", "generation", "fuel", "transmission",
    "drive", "color", "wheel", "seller_type", "city", "region",
]

features = num_features + bool_features + cat_features
# убрать дубликаты, если есть
features = list(dict.fromkeys(features))

# оставить только те, что реально есть
features = [f for f in features if f in df.columns]
cat_features = [f for f in cat_features if f in df.columns]

print(f"features: {len(features)} (num={len(num_features)}, "
      f"bool={len(bool_features)}, cat={len(cat_features)})")

# ── таргет в log ──────────────────────────────────────────────────────
df = df.dropna(subset=[target]).copy()
y = np.log1p(df[target])
X = df[features]

# ── split ─────────────────────────────────────────────────────────────
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ── CatBoost ──────────────────────────────────────────────────────────
model = CatBoostRegressor(
    iterations=3000,
    learning_rate=0.05,
    depth=8,
    l2_leaf_reg=3.0,
    cat_features=cat_features,
    random_seed=42,
    verbose=200,
    early_stopping_rounds=150,
    eval_metric="MAE",
)

model.fit(
    X_tr, y_tr,
    eval_set=(X_te, y_te),
    use_best_model=True,
)

# ── метрики ───────────────────────────────────────────────────────────
pred_log = model.predict(X_te)
pred_rub = np.expm1(pred_log)
y_te_rub = np.expm1(y_te)

mae_log = mean_absolute_error(y_te, pred_log)
r2_log = r2_score(y_te, pred_log)
mae_rub = mean_absolute_error(y_te_rub, pred_rub)
rmse_rub = np.sqrt(((y_te_rub - pred_rub) ** 2).mean())
mape = np.mean(np.abs((y_te_rub - pred_rub) / y_te_rub)) * 100

print(f"\n─── CatBoost ───")
print(f"MAE (log)  : {mae_log:.4f}")
print(f"R²  (log)  : {r2_log:.4f}")
print(f"MAE (руб)  : {mae_rub:,.0f}")
print(f"RMSE (руб) : {rmse_rub:,.0f}")
print(f"MAPE       : {mape:.2f}%")

# ── feature importance ────────────────────────────────────────────────
imp = pd.DataFrame({
    "feature": features,
    "importance": model.get_feature_importance(),
}).sort_values("importance", ascending=False)

print("\nТоп-25 фич:")
print(imp.head(25).to_string(index=False))

# ── сохранить модель ──────────────────────────────────────────────────
model.save_model(MODELS_DIR / "catboost_v1.cbm")
print(f"\n[model saved] {MODELS_DIR / 'catboost_v1.cbm'}")

# ── сохранить предсказания для анализа ────────────────────────────────
pred_df = pd.DataFrame({
    "y_true_log": y_te,
    "y_pred_log": pred_log,
    "y_true_rub": y_te_rub,
    "y_pred_rub": pred_rub,
    "abs_err_rub": np.abs(y_te_rub - pred_rub),
}).reset_index(drop=True)
pred_df.to_parquet(DATA_RAW.parent / "processed" / "preds_v1.parquet", index=False)
print(f"[preds saved]")

shape: (30152, 71)
features: 69 (num=9, bool=49, cat=11)
0:	learn: 0.6760636	test: 0.6647715	best: 0.6647715 (0)	total: 217ms	remaining: 10m 51s
200:	learn: 0.1496329	test: 0.1541948	best: 0.1541948 (200)	total: 16.4s	remaining: 3m 47s
400:	learn: 0.1309802	test: 0.1396491	best: 0.1396491 (400)	total: 33.1s	remaining: 3m 34s
600:	learn: 0.1204319	test: 0.1333193	best: 0.1333190 (599)	total: 42.8s	remaining: 2m 51s
800:	learn: 0.1135351	test: 0.1300538	best: 0.1300538 (800)	total: 52.1s	remaining: 2m 23s
1000:	learn: 0.1080505	test: 0.1279396	best: 0.1279396 (1000)	total: 1m	remaining: 2m 1s
1200:	learn: 0.1029350	test: 0.1262187	best: 0.1262187 (1200)	total: 1m 9s	remaining: 1m 43s
1400:	learn: 0.0986569	test: 0.1250712	best: 0.1250712 (1400)	total: 1m 18s	remaining: 1m 29s
1600:	learn: 0.0948247	test: 0.1240384	best: 0.1240384 (1600)	total: 1m 26s	remaining: 1m 15s
1800:	learn: 0.0915963	test: 0.1234561	best: 0.1234517 (1798)	total: 1m 36s	remaining: 1m 4s
2000:	learn: 0.0886694	test:

In [2]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from src.db import DATA_RAW
from sklearn.metrics import mean_absolute_error, r2_score
from catboost import CatBoostRegressor

df = pd.read_parquet(DATA_RAW / "cars.parquet")

# features как в прошлом ноутбуке
num_features = ["car_age", "mileage_km", "mileage_per_year", "power_hp",
                "engine_volume_l", "owners_count", "photos_count",
                "seller_age_years"]
bool_features = df.select_dtypes(include="bool").columns.tolist()
cat_features = ["brand", "model", "generation", "fuel", "transmission",
                "drive", "color", "wheel", "seller_type", "city", "region"]
features = num_features + bool_features + cat_features
features = [f for f in features if f in df.columns]
cat_features = [f for f in cat_features if f in df.columns]

y = np.log1p(df["price"])

# ── TIME SPLIT: обучаем на 2022 и старше, тест на 2023+ ──────────────
mask_train = df["year"] <= 2022
mask_test = df["year"] >= 2023

X_tr, y_tr = df.loc[mask_train, features], y[mask_train]
X_te, y_te = df.loc[mask_test, features], y[mask_test]

print(f"train: {len(X_tr)}  test: {len(X_te)}")
print(f"train year range: {df.loc[mask_train,'year'].min()}–{df.loc[mask_train,'year'].max()}")
print(f"test  year range: {df.loc[mask_test,'year'].min()}–{df.loc[mask_test,'year'].max()}")

model = CatBoostRegressor(
    iterations=3000, learning_rate=0.05, depth=8,
    cat_features=cat_features, random_seed=42,
    verbose=500, early_stopping_rounds=150, eval_metric="MAE",
)
model.fit(X_tr, y_tr, eval_set=(X_te, y_te), use_best_model=True)

pred_log = model.predict(X_te)
pred_rub = np.expm1(pred_log)
y_te_rub = np.expm1(y_te)

print(f"\n─── Time split ───")
print(f"MAE (log)  : {mean_absolute_error(y_te, pred_log):.4f}")
print(f"R²  (log)  : {r2_score(y_te, pred_log):.4f}")
print(f"MAE (руб)  : {mean_absolute_error(y_te_rub, pred_rub):,.0f}")
print(f"MAPE       : {np.mean(np.abs((y_te_rub - pred_rub) / y_te_rub)) * 100:.2f}%")

train: 22281  test: 7871
train year range: 1999–2022
test  year range: 2023–2026
0:	learn: 0.6243486	test: 0.8992739	best: 0.8992739 (0)	total: 42.6ms	remaining: 2m 7s
500:	learn: 0.1319364	test: 0.2052032	best: 0.2051530 (499)	total: 21.5s	remaining: 1m 47s
1000:	learn: 0.1162249	test: 0.1933088	best: 0.1933088 (1000)	total: 42.4s	remaining: 1m 24s
1500:	learn: 0.1059003	test: 0.1901386	best: 0.1900335 (1499)	total: 1m 3s	remaining: 1m 3s
Stopped by overfitting detector  (150 iterations wait)

bestTest = 0.1894790781
bestIteration = 1628

Shrink model to first 1629 iterations.

─── Time split ───
MAE (log)  : 0.1895
R²  (log)  : 0.8965
MAE (руб)  : 1,100,479
MAPE       : 17.76%
